# Conversational search contract

Issue #37. Define a small, inspectable offline regression set before building the model adapter. The current catalog supports historical sales in Durham County, filtered only by minimum/maximum sold price, minimum beds/baths, and ZIP. Map area stays a separate user-controlled filter. Missing or vague values need a question; unsupported attributes must never be silently dropped. No provider request or private query is made in this notebook.

In [1]:
import json
from collections import Counter
from pathlib import Path

cases = json.loads(
    Path("tests/fixtures/search_intent_cases.json").read_text(encoding="utf-8")
)
allowed = {"min_price", "max_price", "min_beds", "min_baths", "zip"}
assert len(cases) == len({case["id"] for case in cases})
assert all(
    case["query"].strip() and case["status"] in {"ready", "clarify", "unsupported"}
    for case in cases
)
assert all(set(case["filters"]) <= allowed for case in cases)
assert all(case["filters"] for case in cases if case["status"] == "ready")
assert all(not case["filters"] for case in cases if case["status"] != "ready")
print(
    json.dumps(
        {
            "cases": len(cases),
            "status_counts": dict(Counter(case["status"] for case in cases)),
            "allowed_filters": sorted(allowed),
        },
        indent=2,
    )
)

{
  "cases": 13,
  "status_counts": {
    "ready": 4,
    "clarify": 2,
    "unsupported": 7
  },
  "allowed_filters": [
    "max_price",
    "min_baths",
    "min_beds",
    "min_price",
    "zip"
  ]
}


The regression set is a contract, not evidence of model accuracy. Live provider evaluation must compare responses to these expectations when an API key is available. The server must independently validate every returned field and refuse unsupported or conflicting requests; only the existing repository search builds SQL. Model outputs must not create property descriptions or facts. A catalog-only description may be assembled deterministically from verified sale fields.